# GPU-Solver Colab Watch (Git-우편함)
터널 없음. cmd/result JSON을 git repo로 비동기 교환.

**연막**: execute_request=stub. REQ 받으면 NotImplementedError RES 뱉음. 골격 왕복만 실측.

**선결**: Colab Secrets에 `GPU_MAILBOX_TOKEN` = fine-grained PAT(repo scope) 저장.
사이드바 🔑 → 시크릿 추가 → 노트북 접근 ON. 셀2가 `userdata.get`으로 읽음 (평문 노출 0).

In [ ]:
# 1) GPU 확인
!nvidia-smi -L

In [ ]:
# 2) git 인증 + clone (PAT = Colab Secrets에서)
import subprocess
from google.colab import userdata

PAT = userdata.get("GPU_MAILBOX_TOKEN")   # 평문 노출 0. Secrets에 저장 필요.
USER = "alexxony"

def sh(args):
    return subprocess.run(args, capture_output=True, text=True)

# 런타임 재시작 시 휘발 → 매번 재실행 필수 (pull 머지 ident).
sh(["git", "config", "--global", "user.email", "colab@gpu-solver"])
sh(["git", "config", "--global", "user.name", "colab-watch"])

def clone(repo):
    base = "github.com/" + USER + "/" + repo
    url = "https://" + USER + ":" + PAT + "@" + base + ".git"
    dst = "/content/" + repo
    sh(["rm", "-rf", dst])
    r = sh(["git", "clone", "-q", url, dst])
    ok = "OK" if r.returncode == 0 else r.stderr
    print(repo, "->", ok)

clone("gpu-mailbox")
clone("gpu-solver-loop")

In [ ]:
# 2.5) [임시·폐기] git pull 헬스체크 — stderr 확보 (§177 진단). 통과 확인 후 셀 삭제.
import subprocess
r = subprocess.run(["git", "-C", "/content/gpu-mailbox", "pull", "--no-edit"],
                   capture_output=True, text=True)
print("RC:", r.returncode)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr)
# RC==0 이면 prior blocker 해소. 비0이면 STDERR가 진범(ident/auth/divergent).

In [ ]:
# 3) watch.py 골격 self-check (GPU 없이 로직)
#    watch.py는 repo 루트 아니라 loop/ 안에 있음.
import sys
sys.path.insert(0, "/content/gpu-solver-loop/loop")
!cd /content/gpu-solver-loop/loop && python watch.py --selfcheck

In [ ]:
# 4) watch — REQ 받으면 executor 위임(진짜 GPU 라운드). reload 금지(§206 import 충돌).
#    순서(§s2-203): 로컬서 REQ push 먼저 → 그 다음 이 셀. 그래야 1틱째 바로 잡음.
import sys
sys.path.insert(0, "/content/gpu-solver-loop/loop")
import watch

MB = "/content/gpu-mailbox"
print("watch 시작 — poll 5s, max 120틱(~10분). 로컬 runner 돌려.")
watch.watch_loop(MB, poll_s=5.0, max_iters=120)
print("watch 종료. result/ 확인:")
!ls -la /content/gpu-mailbox/result/